# 第110章 PCA降维与可视化

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 25 / 34 步：深入客户分群与降维表达**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 层次聚类与DBSCAN  →  **本章任务：** PCA降维与可视化  →  **下一步：** 交叉验证策略
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

高维数据（比如葡萄酒数据里的 13 个检测指标）又难画又难看，点与点挤在一起，规律往往藏在多个维度的组合里。


## 本章目标

学完本章，你将能够：

- **理解**：理解「PCA降维与可视化」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「PCA降维与可视化」的关键输出指标。
- **迁移**：能把「PCA降维与可视化」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：高维数据（比如葡萄酒数据里的 13 个检测指标）又难画又难看，点与点挤在一起，规律往往藏在多个维度的组合里。PCA 把这些指标重新拧成少数几个综合方向（主成分），在保留大部分信息的同时把数据降到你一眼能画的 2 维或 3 维。这样不仅能直接在散点图上看出类别是否分开，也让后续建模更快、更稳；从用户画像到基因表达，降维几乎总是排在第一位的那一步探索动作。


- PCA：\(\max_{||w||=1}Var(Xw)\)
- 主成分是协方差矩阵特征向量
- 特征值对应解释方差
- PCA 保留方差，不保证保留目标信息（打个比方：把几十项指标拧成少数几个“综合分”，既好画图又省事；但综合分只按它自己有多“分散”来排，未必就是最该预测的那几项。）


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 数据与问题定义 | `.fit_transform()`、`.fit()` | 先明确样本、特征、目标和验证方式，再训练模型。 | 切分前拟合 PCA |
| 模型、公式与诊断 | `np.cumsum()`、`np.argmax()`、`pd.DataFrame()`、`loadings.abs()` | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | 未标准化不同量纲特征 |


## 例 1｜数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


<!-- math-foundation:chapter-110 -->
### 数学推导｜PCA 寻找方差最大的正交方向

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜先中心化数据并计算协方差。** $\Sigma=X_c^TX_c/(n-1)$。

**第 2 步｜寻找单位方向 $v$ 上方差最大者。** 投影 $z=X_cv$ 的方差为 $v^T\Sigma v$，所以求解

$$
\max_{\lVert v\rVert=1}v^T\Sigma v
$$

**第 3 步｜使用拉格朗日乘子。** 对 $v^T\Sigma v-\lambda(v^Tv-1)$ 求导，得到 $\Sigma v=\lambda v$。最大特征值对应第一主成分，后续方向再加与前面方向正交的约束。

**第 4 步｜用特征值计算解释方差比。** $EVR_k=\lambda_k/\sum_j\lambda_j$。

**把上面的关系收束为本章计算式：**

$$
\Sigma v_k=\lambda_kv_k,\qquad z_k=Xv_k,\qquad EVR_k=\frac{\lambda_k}{\sum_j\lambda_j}
$$

**符号解释：** $v_k$ 是第 $k$ 个主成分方向，$\lambda_k$ 是其解释方差。

**代码对应：** 标准化后拟合 `PCA`，用 `explained_variance_ratio_` 决定保留维度。

**使用边界：** PCA 是无监督线性投影；高方差方向不一定最有业务或预测价值。


In [ ]:
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

data = load_wine(as_frame=True)
X, y = data.data, data.target
Xs = StandardScaler().fit_transform(X)
pca = PCA().fit(Xs)


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：PCA 的作用是把多个特征压缩成少数几个主成分。现在请你修改示例 1 里的 `PCA()`，让它只保留 **前 2 个主成分**（`n_components=2`），并打印这两个主成分的**累计解释方差**。想一想：只留 2 个成分时，解释方差还能不能接近 95%？请在下方补全两个填空，然后运行自检验证结果。


In [ ]:
try:
    # 请在下方填写代码
    # 任务：修改示例 1 的 PCA，只保留前 2 个主成分，并打印累计解释方差。
    import numpy as np

    # 填空 1：新建 PCA，只保留 2 个主成分
    # pca2 = PCA(____)   # ← 补全后取消注释

    pca2.fit(Xs)

    # 填空 2：计算前 2 个主成分的累计解释方差
    # cum2 = float(____)   # ← 补全后取消注释

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
import numpy as np
import pandas as pd

cumulative = np.cumsum(pca.explained_variance_ratio_)
needed = int(np.argmax(cumulative >= 0.95) + 1)
print("95%方差成分数:", needed, "累计:", round(cumulative[needed - 1], 3))
loadings = pd.DataFrame(
    pca.components_[:2].T, index=X.columns, columns=["PC1", "PC2"]
)
display(loadings.abs().sort_values("PC1", ascending=False).head())


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

_demo_data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in _demo_data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 切分前拟合 PCA
- 未标准化不同量纲特征
- 把主成分命名为未经验证的潜变量
- 认为解释方差越高预测一定越好


## 练习与作业

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 110.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 110.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 110.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

使用 PCA 压缩高维数据，分析解释方差、特征载荷和降维后的分类性能。


### 你已经掌握

- 计算累计解释方差
- 理解特征值和主成分
- 解释载荷
- 把 PCA 放入无泄漏流水线


### 需要注意

- 切分前拟合 PCA
- 未标准化不同量纲特征
- 把主成分命名为未经验证的潜变量
- 认为解释方差越高预测一定越好


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
import numpy as np

# 答案：只保留前 2 个主成分，观察累计解释方差
pca2 = PCA(n_components=2).fit(Xs)
cum2 = float(np.sum(pca2.explained_variance_ratio_))

print("前2个主成分累计解释方差:", round(cum2, 3))


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error

practice_model = make_pipeline(StandardScaler(), PCA(n_components=2), Ridge())
practice_scores = cross_val_score(
    practice_model, X, y, cv=3, scoring="neg_mean_absolute_error"
)
practice_score = float(practice_scores.mean())
print("负MAE均值（越接近0越好）:", round(practice_score, 3))
